# Fine-tune BioLinkBERT Cross-Encoder Re-Ranker

Trains a cross-encoder on (query, trial) pairs using `BinaryCrossEntropyLoss`.
The cross-encoder re-ranks hybrid retrieval candidates for better NDCG.

**Setup:**
1. Upload `train_pairs.jsonl` and `val_pairs.jsonl` to Google Drive
2. Runtime > Change runtime type > **A100 GPU** (or T4)
3. Run all cells

**What you get back:**
- `fine-tuned-cross-encoder.zip` — download to local `models/cross-encoder/fine-tuned/`

## Step 1: Verify GPU and install packages

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
!pip install -q --upgrade sympy sentence-transformers datasets pyyaml

## Step 2: Load data from Google Drive

1. Upload `train_pairs.jsonl` and `val_pairs.jsonl` to your Google Drive root (or a subfolder)
2. Update `DRIVE_FOLDER` below if you used a subfolder

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive"  # update if files are in a subfolder

os.makedirs("data/training", exist_ok=True)
!cp "{DRIVE_FOLDER}/train_pairs.jsonl" data/training/
!cp "{DRIVE_FOLDER}/val_pairs.jsonl" data/training/

# Verify
for f in ["data/training/train_pairs.jsonl", "data/training/val_pairs.jsonl"]:
    size = os.path.getsize(f) / 1e6
    print(f"  {f}: {size:.1f} MB")

## Step 3: Convert triplets to binary pairs and load model

In [ ]:
import json
import random
import logging

from datasets import Dataset
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from sentence_transformers.cross_encoder.trainer import CrossEncoderTrainer
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
# Config — tuned for T4 GPU (~2 hours)
CONFIG = {
    "model_name": "michiyasunaga/BioLinkBERT-base",
    "num_labels": 1,
    "max_length": 512,
    "epochs": 1,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "fp16": True,
    "logging_steps": 100,
    "eval_steps": 2000,
    "save_steps": 2000,
    "save_total_limit": 3,
    "max_train_triplets": 100_000,  # 100K triplets → 200K pairs
    "max_val_triplets": 5_000,      # 5K triplets → 10K pairs (fast val loss)
    "max_eval_samples": 1_000,      # 1K samples for CERerankingEvaluator
    "early_stopping_patience": 3,
    "output_dir": "models/cross-encoder-fine-tuned",
}

In [ ]:
# Load cross-encoder model
model = CrossEncoder(
    CONFIG["model_name"],
    num_labels=CONFIG["num_labels"],
    max_length=CONFIG["max_length"],
    device="cuda",
)
print(f"Model: {CONFIG['model_name']}")
print(f"num_labels: {CONFIG['num_labels']}, max_length: {CONFIG['max_length']}")

In [ ]:
def load_triplets_as_pairs(filepath: str, max_triplets: int | None = None) -> Dataset:
    """Convert JSONL triplets to binary-labeled pairs, with optional subsampling."""
    triplets = []
    with open(filepath) as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            query = rec.get("query", "")
            positive = rec.get("positive", "")
            negative = rec.get("negative", "")
            if not query or not positive or not negative or not negative.strip():
                continue
            triplets.append(rec)

    print(f"  Loaded {len(triplets):,} triplets from {filepath}")

    if max_triplets and len(triplets) > max_triplets:
        rng = random.Random(42)
        triplets = rng.sample(triplets, max_triplets)
        print(f"  Subsampled to {len(triplets):,} triplets")

    rows = []
    for rec in triplets:
        rows.append({"sentence1": rec["query"], "sentence2": rec["positive"], "label": 1.0})
        rows.append({"sentence1": rec["query"], "sentence2": rec["negative"], "label": 0.0})

    return Dataset.from_list(rows)

train_ds = load_triplets_as_pairs(
    "data/training/train_pairs.jsonl",
    max_triplets=CONFIG["max_train_triplets"],
)
val_ds = load_triplets_as_pairs(
    "data/training/val_pairs.jsonl",
    max_triplets=CONFIG["max_val_triplets"],
)

n_pos = sum(1 for r in train_ds if r["label"] == 1.0)
print(f"\nTrain: {len(train_ds):,} pairs ({n_pos:,} pos, {len(train_ds)-n_pos:,} neg)")
print(f"Val:   {len(val_ds):,} pairs")

steps = len(train_ds) // CONFIG["batch_size"] * CONFIG["epochs"]
est_hours = steps * 0.65 / 3600
print(f"Estimated: {steps:,} steps, ~{est_hours:.1f} hours on T4")

In [ ]:
# Data quality check: 3 positive + 3 negative examples
positives = [r for r in train_ds if r["label"] == 1.0]
negatives = [r for r in train_ds if r["label"] == 0.0]

print("--- 3 POSITIVE examples (label=1.0) ---")
for i, row in enumerate(positives[:3]):
    print(f"\n  [{i+1}] Query: {row['sentence1'][:100]}")
    print(f"      Trial: {row['sentence2'][:120]}...")

print("\n--- 3 NEGATIVE examples (label=0.0) ---")
for i, row in enumerate(negatives[:3]):
    print(f"\n  [{i+1}] Query: {row['sentence1'][:100]}")
    print(f"      Trial: {row['sentence2'][:120]}...")

## Step 4: Build evaluator

In [ ]:
# Build CERerankingEvaluator from val triplets (subsampled for speed)
val_rows = []
with open("data/training/val_pairs.jsonl") as f:
    for line in f:
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        if not row.get("negative", "").strip():
            continue
        val_rows.append(row)

rng = random.Random(42)
if len(val_rows) > CONFIG["max_eval_samples"]:
    val_rows = rng.sample(val_rows, CONFIG["max_eval_samples"])

samples = [
    {"query": r["query"], "positive": [r["positive"]], "negative": [r["negative"]]}
    for r in val_rows
]

evaluator = CERerankingEvaluator(
    samples=samples,
    name="CERerankingEvaluator",
    batch_size=64,
)
print(f"CE reranking evaluator: {len(samples)} samples")

## Step 5: Train

In [ ]:
from transformers import EarlyStoppingCallback

loss = BinaryCrossEntropyLoss(model=model)

training_args = CrossEncoderTrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=CONFIG["fp16"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    logging_steps=CONFIG["logging_steps"],
    load_best_model_at_end=True,
    metric_for_best_model="CERerankingEvaluator_ndcg@10",
    greater_is_better=True,
    report_to="none",
)

early_stop = EarlyStoppingCallback(
    early_stopping_patience=CONFIG["early_stopping_patience"],
)

steps_per_epoch = len(train_ds) // CONFIG["batch_size"]
total_steps = steps_per_epoch * CONFIG["epochs"]
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"Steps/epoch: {steps_per_epoch:,}")
print(f"Total steps: {total_steps:,}")
print(f"Early stopping: patience={CONFIG['early_stopping_patience']} evals on NDCG@10")
print(f"Eval every {CONFIG['eval_steps']} steps → max {total_steps // CONFIG['eval_steps']} evals before stop")

In [ ]:
import time

trainer = CrossEncoderTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
    evaluator=evaluator,
    callbacks=[early_stop],
)

start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"\nTraining completed in {elapsed / 60:.1f} minutes")
if trainer.state.global_step < total_steps:
    print(f"Early stopped at step {trainer.state.global_step} / {total_steps}")

## Step 6: Evaluate and save

In [ ]:
# Final evaluation
eval_results = evaluator(model)

print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)
for key in sorted(eval_results):
    print(f"  {key}: {eval_results[key]:.4f}")

ndcg = eval_results.get("CERerankingEvaluator_ndcg@10", "N/A")
map_score = eval_results.get("CERerankingEvaluator_map", "N/A")
mrr_score = eval_results.get("CERerankingEvaluator_mrr@10", "N/A")

print(f"\nSummary:")
print(f"  NDCG@10:  {ndcg}")
print(f"  MAP:      {map_score}")
print(f"  MRR@10:   {mrr_score}")

In [ ]:
from datetime import datetime, timezone

# Save model
model.save(CONFIG["output_dir"])
print(f"Model saved to {CONFIG['output_dir']}/")

# Save metadata
metadata = {
    "training_date": datetime.now(timezone.utc).isoformat(),
    "base_model": CONFIG["model_name"],
    "model_type": "cross-encoder",
    "num_labels": CONFIG["num_labels"],
    "dataset_size": {"train": len(train_ds), "val": len(val_ds)},
    "hyperparameters": {
        "epochs": CONFIG["epochs"],
        "batch_size": CONFIG["batch_size"],
        "learning_rate": CONFIG["learning_rate"],
        "warmup_ratio": CONFIG["warmup_ratio"],
        "weight_decay": CONFIG["weight_decay"],
        "loss": "BinaryCrossEntropyLoss",
        "max_length": CONFIG["max_length"],
        "fp16": CONFIG["fp16"],
    },
    "device": "cuda",
    "gpu": torch.cuda.get_device_name(0),
    "training_time_minutes": round(elapsed / 60, 1),
    "eval_metrics": eval_results,
}

with open(f"{CONFIG['output_dir']}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata saved.")

## Step 7: Download the fine-tuned model

Saves the model to Google Drive as a zip. On your local machine:
1. Download `fine-tuned-cross-encoder.zip` from Drive
2. Unzip to `models/cross-encoder/fine-tuned/`
3. Evaluate: `python scripts/evaluate_cross_encoder.py --model models/cross-encoder/fine-tuned`
4. Demo: `python scripts/demo_reranker.py --model models/cross-encoder/fine-tuned`

In [ ]:
import shutil

# Zip the model directory
zip_path = shutil.make_archive("fine-tuned-cross-encoder", "zip", CONFIG["output_dir"])
zip_size = os.path.getsize(zip_path) / 1e6
print(f"Zipped model: {zip_path} ({zip_size:.1f} MB)")

# Save to Google Drive
drive_dest = "/content/drive/MyDrive/fine-tuned-cross-encoder.zip"
shutil.copy(zip_path, drive_dest)
print(f"Saved to Google Drive: {drive_dest}")
print("\nDownload this file from Drive, then unzip to models/cross-encoder/fine-tuned/")